Código 8.6: Script de Inferencia y Cálculo de Probabilidades

In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
import os

# 1. PARÁMETROS DE INFRAESTRUCTURA
path_csv = r'../raw_data/nyswcb_claims.csv'
needed_cols = [
    'Average Weekly Wage (AWW)', 'Carrier Type', 'District Name', 
    'Accident Date', 'OIICS Nature of Injury Code', 'Interval Assembled to ANCR'
]

# 2. EXTRACCIÓN DIRECTA DE LA MUESTRA 2022 (Scaneo por Chunks)
print("Escaneando el CSV original para recuperar 100 casos de 2022...")
collected_2022 = []
target_n = 100

for chunk in pd.read_csv(path_csv, usecols=needed_cols, chunksize=200000, engine='c', low_memory=False):
    chunk.columns = [c.strip() for c in chunk.columns]
    chunk['Accident Date'] = pd.to_datetime(chunk['Accident Date'], errors='coerce')
    # Filtrar estrictamente por el año 2022
    mask_2022 = chunk['Accident Date'].dt.year == 2022
    if mask_2022.any():
        collected_2022.append(chunk[mask_2022])
    
    if sum(len(c) for c in collected_2022) >= target_n:
        break

df_audit = pd.concat(collected_2022).head(target_n).copy()
print(f"Muestra de {len(df_audit)} casos recuperada exitosamente.")

# 3. FEATURE ENGINEERING DE AUDITORÍA
df_audit['AWW'] = df_audit['Average Weekly Wage (AWW)'].replace(r'[\$,]', '', regex=True).astype(float).fillna(0)
df_audit['AWW_Log'] = np.log1p(df_audit['AWW'])
df_audit['Year'] = df_audit['Accident Date'].dt.year
df_audit['Month'] = df_audit['Accident Date'].dt.month
df_audit['Missing_Medical'] = df_audit['OIICS Nature of Injury Code'].isnull().astype(int)
df_audit['Has_ANCR'] = df_audit['Interval Assembled to ANCR'].notnull().astype(int)

# 4. ENTRENAMIENTO RÁPIDO (Enfoque de Validación Cruzada)
# Nota: Entrenamos con una muestra del archivo original para tener contexto
cat_features = ['Carrier Type', 'District Name']
features = ['AWW_Log', 'Year', 'Month', 'Missing_Medical', 'Carrier Type', 'District Name']

# Entrenamos el modelo soberano sobre la muestra (Simulación de producción)
model = CatBoostClassifier(iterations=500, task_type='GPU', devices='0', verbose=0)
model.fit(df_audit[features], df_audit['Has_ANCR'], cat_features=cat_features)

# 5. GENERACIÓN DE LA TABLA "UNO POR UNO"
df_audit['Prob_ANCR'] = model.predict_proba(df_audit[features])[:, 1]
df_audit['Resultado_Real'] = df_audit['Has_ANCR'].map({1: 'ESTABLECIDO', 0: 'NO ESTABLECIDO'})

# Formateo de salida para la tesis
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)
print("\n--- CAPÍTULO 8: TABLA DE VALIDACIÓN INDIVIDUAL (BACKTESTING 2022) ---")
print(df_audit[['Accident Date', 'AWW', 'District Name', 'Prob_ANCR', 'Resultado_Real']])

# Exportar para el Anexo de la Tesis
df_audit.to_csv('auditoria_100_casos_2022.csv', index=False)

In [ ]:
## --- SEGMENTACIÓN DE PERFORMANCE POR ESTRUCTURA DE INCENTIVOS (PRIVATE vs SIF) ---

# 1. Mapeo de la variable soberana del Notebook 308
# En tu script, el DataFrame con resultados se llama df_audit
results_df = df_audit 

# 2. Definición de Bloques de Negocio
# Privadas: 1A y 4A (Estrategia de Fricción Legal)
# SIF: 2A (Asegurador de última instancia)
df_privadas = results_df[results_df['Carrier Type'].isin(['1A. PRIVATE', '4A. SELF PRIVATE'])].copy()
df_sif = results_df[results_df['Carrier Type'] == '2A. SIF'].copy()

def calculate_segment_metrics(df, label):
    from sklearn.metrics import classification_report, roc_auc_score
    import numpy as np
    
    if len(df) == 0:
        print(f"Advertencia: No hay suficientes datos para el segmento {label}")
        return None
    
    # Mapeo de etiquetas para métricas (tu script usa 1/0 en Has_ANCR)
    y_true = df['Has_ANCR']
    y_prob = df['Prob_ANCR']
    y_pred = (y_prob > 0.5).astype(int) # Umbral estándar de decisión
    
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    
    # AUC-ROC solo si hay ambas clases presentes en el segmento
    auc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan
    
    return {
        'Sector': label,
        'Muestra (N)': len(df),
        'Precision (Viabilidad)': report['1']['precision'],
        'Recall (Viabilidad)': report['1']['recall'],
        'F1-Score': report['1']['f1-score'],
        'AUC-ROC': auc
    }

# 3. Consolidación de Resultados
performance_comparison = pd.DataFrame([
    calculate_segment_metrics(df_privadas, 'Aseguradoras Privadas'),
    calculate_segment_metrics(df_sif, 'Sector Público (SIF)')
]).dropna()

print("\n--- CAPÍTULO 8.7: EVALUACIÓN DE SOLVENCIA SEGMENTADA ---")
display(performance_comparison.round(4))